# Anatomy of a Tool Call

> 📓 *Companion to* **Modern Agentic AI Engineer** *· Ch 12 §12.1–12.2 · type: concept-lab*

You're about to watch a single tool call happen in slow motion. One prompt, one
round-trip, and by the end you'll be able to name every moving part: the tool
**schema** you send, the model's **tool-use request**, your **executor**, and the
**tool-result** you send back. No framework — just the raw shapes.

## 🧠 Why this matters

A language model can only emit text. It cannot read your database, hit an API, or
run a calculation — until you give it *tools*. Tool use is the single capability
that turns a chatbot into an agent, and every later topic in this book (RAG,
memory, multi-agent, MCP) is layered on the loop you meet here.

The key idea that everything hangs on: **the model never executes anything.** You
hand it a list of JSON schemas; if it wants to act, it replies with a structured
*request* to call a tool. **Your** code validates that request, runs the real
function, and feeds the result back. The intelligence is in the room; all the
agency — and the responsibility — stays outside, with you.

Picture the model as a brilliant consultant locked in a room with only a mail
slot. It can reason about anything, but to *act* it must write an instruction on a
card and slide it through the slot. You decide whether to honor the card, do the
work with your own hands, and slide the result back.

## Objectives & prerequisites

**By the end you can:**

- Read a single round-trip of a tool call and name every part.
- Inspect the JSON shape of a tool schema and a `tool_use` block.
- Execute a tool by hand and feed a `tool_result` back to get the final answer.
- Spot why a vague tool description sabotages the whole loop.

**Prereqs:** Chapter 11 (model APIs / SDK shapes); the notebooks under
`learn/part-03-llm-substrate/11-*`. No prior tool-use experience needed.

**Run first:** the Setup cell below. It defaults to `MOCK=1`, so this notebook
runs **free and offline with no API key**.

In [ ]:
# --- Setup -------------------------------------------------------------------
import json
import os
import random

from dotenv import load_dotenv

load_dotenv()  # loads a local .env if present; never hardcode keys

# MOCK=1 (default) returns canned, realistic tool-use blocks so this notebook
# runs free, offline, and deterministically. Set COMPANION_MOCK=0 to hit the
# live Anthropic API (costs ~2 short calls; needs ANTHROPIC_API_KEY).
MOCK = os.getenv("COMPANION_MOCK", "1") == "1"

random.seed(12)  # determinism for anything stochastic in this notebook

MODEL = "claude-opus-4-8"  # the book's default; latest, most capable

if MOCK:
    print("MOCK mode: canned responses. No API key needed, nothing is billed.")
else:
    if not os.getenv("ANTHROPIC_API_KEY"):
        raise RuntimeError(
            "COMPANION_MOCK=0 but ANTHROPIC_API_KEY is not set. "
            "Add it to your environment/.env or set COMPANION_MOCK=1."
        )
    import anthropic  # only imported on the live path

    client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY
    print(f"LIVE mode: calling {MODEL}. Two short calls will be billed.")

## 1 · One tiny tool: a calculator

A tool is three things: a **name**, a **description**, and a **JSON Schema** for
its inputs. That's it. The description is not decoration — it's the *primary
interface*, because the model picks tools by reading it. Say what the tool does
and, just as importantly, *when to call it*.

Here's a deliberately tiny tool so the shape is the whole lesson: a calculator
that evaluates a basic arithmetic expression.

In [ ]:
CALCULATOR = {
    "name": "calculator",
    "description": (
        "Evaluate a basic arithmetic expression and return the numeric result. "
        "Call this whenever the user asks for a calculation or any exact "
        "arithmetic; do not do multi-digit math in your head."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "expression": {
                "type": "string",
                "description": (
                    "A single arithmetic expression using + - * / ( ) and "
                    "numbers, e.g. '128 * 974'."
                ),
            },
        },
        "required": ["expression"],
    },
}

# This is just data -- a dict. Look at its shape: name, description, input_schema.
print(json.dumps(CALCULATOR, indent=2))

The `input_schema` is plain JSON Schema. `type: object`, a `properties` map, and a
`required` list. The model reads this and learns exactly what arguments it's
allowed to send. Notice the description tells it *when* to reach for the tool —
"do not do multi-digit math in your head" — which is the kind of guidance that
makes an agent feel sharp instead of hapless.

## 2 · 🔮 Predict: will the model answer, or ask to call the tool?

We'll send the prompt **"What is 128 times 974?"** with the calculator tool
attached.

**Before you run the next cell, predict:** does the model reply with prose (a
guessed number), or does it stop and emit a *request* to call `calculator`? And
if it calls the tool, what `expression` will it pass?

Write your prediction down, then run.

In [ ]:
def mock_first_response():
    """A canned Anthropic-style response: the model requests a tool call.

    This mirrors the real SDK shape closely enough to read and dissect. On the
    live path, the same fields (stop_reason, content blocks with .type) appear
    on the SDK's response object.
    """
    return {
        "stop_reason": "tool_use",
        "content": [
            {
                "type": "text",
                "text": "Let me compute that exactly.",
            },
            {
                "type": "tool_use",
                "id": "toolu_01CalcAbc123",
                "name": "calculator",
                "input": {"expression": "128 * 974"},
            },
        ],
    }


def call_model(messages, tools):
    """Return one model response. MOCK returns a canned dict; live calls the SDK."""
    if MOCK:
        return mock_first_response()
    resp = client.messages.create(
        model=MODEL,
        max_tokens=1024,
        tools=tools,
        messages=messages,
    )
    # Normalize the SDK object into the same dict shape we use in MOCK, so the
    # rest of the notebook reads identically on both paths.
    return {
        "stop_reason": resp.stop_reason,
        "content": [
            {"type": "text", "text": b.text} if b.type == "text"
            else {"type": "tool_use", "id": b.id, "name": b.name, "input": b.input}
            for b in resp.content
        ],
    }


user_prompt = "What is 128 times 974?"
messages = [{"role": "user", "content": user_prompt}]

response = call_model(messages, tools=[CALCULATOR])
print("stop_reason:", response["stop_reason"])
print(json.dumps(response["content"], indent=2))

**What you just saw.** The model did *not* answer with a number. It set
`stop_reason` to `"tool_use"` and returned a `tool_use` block: a request to run
`calculator` with `{"expression": "128 * 974"}`. That stop signal is the heartbeat
of the loop — `tool_use` means "your turn, go execute"; `end_turn` would mean "I'm
done, the text is your answer."

## 3 · Dissect the `tool_use` block

Three fields on a `tool_use` block do all the work. Pull them out so they're not
abstract.

In [ ]:
# Find the tool_use block among the content blocks.
tool_use = next(b for b in response["content"] if b["type"] == "tool_use")

print("id    :", tool_use["id"])     # correlates the request to your result
print("name  :", tool_use["name"])   # which tool to run
print("input :", tool_use["input"])  # the arguments, matching the input_schema

# The id matters: when you send the result back, it MUST carry this exact id so
# the model knows which request you answered. One tool_use_id -> one tool_result.

- **`id`** — a correlation token. Your result must echo it back as `tool_use_id`,
  or the next API call is rejected. One request, one result, matched by id.
- **`name`** — which registered tool to run.
- **`input`** — the arguments the model chose, shaped to your `input_schema`.

The model picked the arguments. It is *generated text* — untrusted input — which
is exactly why your executor validates before it runs anything (you'll build that
discipline in 12-02).

## 4 · Run the tool by hand, feed the result back

Now *you* are the hands outside the room. Execute the requested tool, wrap the
output as a `tool_result` carrying the matching `tool_use_id`, append it to the
conversation, and ask the model again. This time it has the number it needed.

In [ ]:
def calculator(expression: str) -> str:
    """Tiny pure-function tool. Restricted eval: digits and + - * / ( ) only."""
    allowed = set("0123456789+-*/(). ")
    if not set(expression) <= allowed:
        raise ValueError(f"unsupported characters in expression: {expression!r}")
    # eval with no builtins, on a vetted character set, of a tiny expression.
    return str(eval(expression, {"__builtins__": {}}, {}))


# 1) Execute exactly what the model asked for.
result_text = calculator(**tool_use["input"])
print("tool output:", result_text)

# 2) Echo the assistant's turn (with its tool_use block) verbatim into history...
messages.append({"role": "assistant", "content": response["content"]})

# 3) ...then send the result back as a user message, matched by tool_use_id.
tool_result_msg = {
    "role": "user",
    "content": [
        {
            "type": "tool_result",
            "tool_use_id": tool_use["id"],  # MUST match the request's id
            "content": result_text,
        }
    ],
}
messages.append(tool_result_msg)
print(json.dumps(tool_result_msg, indent=2))

In [ ]:
def mock_final_response():
    """Canned second response: the model reads the tool_result and answers."""
    return {
        "stop_reason": "end_turn",
        "content": [
            {"type": "text", "text": "128 times 974 is 124,672."},
        ],
    }


def call_model_again(messages, tools):
    if MOCK:
        return mock_final_response()
    resp = client.messages.create(
        model=MODEL, max_tokens=1024, tools=tools, messages=messages,
    )
    return {
        "stop_reason": resp.stop_reason,
        "content": [
            {"type": "text", "text": b.text} for b in resp.content if b.type == "text"
        ],
    }


final = call_model_again(messages, tools=[CALCULATOR])
print("stop_reason:", final["stop_reason"])  # end_turn -> the loop is done
answer = next(b["text"] for b in final["content"] if b["type"] == "text")
print("final answer:", answer)

**That's the entire protocol.** Prompt → model requests a tool → you execute →
you return a `tool_result` → model produces the final answer. `stop_reason` flips
from `tool_use` to `end_turn`, and the loop ends. Everything else in agent
engineering is this round-trip, repeated and hardened.

## 5 · ⚠️ Pitfall: vague descriptions produce confident misuse

The description *is* the interface. A model choosing tools reads descriptions the
way a literal-minded new hire reads a runbook. Watch how a vague schema invites the
wrong behavior, and how a precise one fixes it.

In [ ]:
BAD_CALCULATOR = {
    "name": "calc",
    "description": "Does math.",  # vague: when? what kind? on what input?
    "input_schema": {
        "type": "object",
        "properties": {"x": {"type": "string"}},  # 'x'? of what?
        "required": ["x"],
    },
}

GOOD_CALCULATOR = CALCULATOR  # the precise one from section 1

print("BAD  :", BAD_CALCULATOR["description"],
      "| param:", list(BAD_CALCULATOR["input_schema"]["properties"]))
print("GOOD :", GOOD_CALCULATOR["description"][:60],
      "...| param:", list(GOOD_CALCULATOR["input_schema"]["properties"]))

# With "Does math." and a parameter named "x", the model has to guess: should it
# even call this for "what is our refund window?" Maybe. Should "x" hold an
# expression, two numbers, words? Different runs guess differently -> your evals
# show baffling inconsistency. The fix is not a smarter model; it is a sharper
# description that says what the tool does, when to call it, and what each
# argument means.

If two tools overlap (`search_kb` and `search_docs`), the model picks between them
essentially at random. Make each tool's territory unambiguous, state *when not* to
use it, and delete tools that earn no calls — an unused tool still costs context
tokens and decision noise on every request.

## 🎯 Senior lens

A junior reads the tool-use API as plumbing to memorize. A senior reads it as a
**security and control boundary**. Because your code sits between the model's
*intent* and the real *execution*, that seam is the one place to put everything
that matters in production: validation, authorization, human approval, rate
limiting, logging. The model proposes; your executor disposes. Design the tool
surface the way you'd review an IAM policy — assume every capability you expose
*will* eventually be called with plausible-but-wrong arguments, and make sure the
worst case is an annoyance, not an incident.

## Recap

- A tool is a **name + description + JSON Schema**; the description is the real
  interface because the model selects tools by reading it.
- The model **never executes** — it returns a `tool_use` request; *your* code runs
  the function and returns a `tool_result`.
- The loop is driven by **`stop_reason`**: `tool_use` → execute and continue;
  `end_turn` → the model is done.
- Every `tool_use` block has an **`id`**; your `tool_result` must echo it as
  `tool_use_id`. One request, one matched result.
- **Vague descriptions cause confident misuse.** Precise, prescriptive schemas are
  the cheapest reliability win you have.

## Exercises

Each exercise *changes* something and asks you to predict the effect before
running. Use the empty cells below. (Solutions live in `solutions/`, added in
Phase 2 — not inline.)

1. **Change the prompt** to something that needs *no* tool, e.g. "Say hello in
   French." In `MOCK`, edit `mock_first_response()` to return `stop_reason:
   "end_turn"` with a text block. Predict: does your loop logic still work when the
   model answers directly without ever calling a tool?
2. **Break the id.** In the `tool_result`, change `tool_use_id` to a wrong value.
   On the live path (`COMPANION_MOCK=0`) the next call errors. Predict the error
   *before* you run it, then read the message — why does the API insist on the
   match?
3. **Add a second tool.** Write a `get_time` tool schema and attach both tools.
   Predict which tool the model calls for "what time is it in Tokyo?" and why a
   precise description is what makes that choice reliable.

In [ ]:
# Exercise 1 -- your code here

In [ ]:
# Exercise 2 -- your code here

In [ ]:
# Exercise 3 -- your code here

## Next

You've read one round-trip by hand. Next you'll **automate** it into a real loop.

- ▶️ **Next notebook:** [`12-02-tool-loop-from-scratch.ipynb`](./12-02-tool-loop-from-scratch.ipynb)
  — build a reusable `while`-loop agent that registers tools, routes requests, and
  stops safely.
- 🔧 **Blueprint (the production version):**
  [`../../../blueprints/agent-loop/`](../../../blueprints/agent-loop/) — typed
  tools, retries, telemetry hooks. You're building the toy; that's the real one.
- 🎓 **Capstone:** this feeds `capstone/agents/raw/` (the support agent) and
  checkpoint `checkpoints/ch12-tool-loop`.
- 📖 **Book:** see §12.1 (the loop) and §12.2 (designing good tools).